In [276]:
import pandas as pd
import numpy as np
import subprocess
import os
from datetime import datetime, timedelta, time, date
import datetime
from datetime import datetime
import locale
import seaborn as sns
import matplotlib.pyplot as plt
import re
import pyproj
import folium

In [277]:
dia = "20260415"                                                                                                                                                         

Paradas

In [278]:
#importar paradas

paradas = pd.read_csv('C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/Paraderos_Zonales_del_SITP.csv')

paradas.head(3)

,X,Y,objectid,cenefa,zona_sitp,nombre,via,direccion_bandera,localidad,longitud,latitud,consecutivo_zona,tipo_m_s,consola,panel,audio,zonas_nuevas,globalid,shape
0,1.001502e+06,1.010205e+06,1,001A00,00,C.C. Iserra 100,AC 100,AC 100 - KR 54,Barrios Unidos,-74.063971,4.688481,001,M,AC 100 - KR 54 (001A00),AC 100 - KR 54,Avenida Calle 100 Carrera 54,C,{1C0DBC4E-15BC-4BBE-BC16-5628077DBE2E},NaN
1,1.003505e+06,1.009719e+06,2,001A01,01,Br. Rincón del Chicó,AC 100,AC 100 - KR 13,Usaquén,-74.045914,4.684091,001,M,AC 100 - KR 13 (001A01),AC 100 - KR 13,Avenida Calle 100 Carrera 13,B,{60A22A44-AD56-4DF4-A3E6-6830B9FB0792},NaN
2,1.001238e+06,1.018098e+06,3,001A02,02,Gimnasio Iragua,AV. Boyacá,AV. Boyacá - AC 170,Suba,-74.066350,4.759867,001,S,AV. Boyacá - AC 170 (001A02),AV. Boyacá - AC 170,Avenida Boyacá Avenida Calle 170,C,{97155274-E4D5-45A4-891F-2DCA19FF10D9},NaN


Matriz distancia

In [279]:
#Zonal

md_zonal = pd.read_csv(f'Z:/01 base_datos/06 matriz_distancia_FMS/{dia}_matriz distancias.csv', encoding='latin')

#Troncal

md_troncal = pd.read_csv(f'Z:/01 base_datos/31 matriz_distancia_troncal_FMS/{dia}_matriz_distancias_troncal.csv', encoding='latin')

md = pd.concat([md_zonal, md_troncal])

md.head(2)

,ï»¿Tipo de Servicio,Id LÃ­nea,LÃ­nea,Configuraciones,ConfiguraciÃ³n Activa,Id SublÃ­nea,Id Ruta,Nombre Ruta,Sentido,Id Nodo,Etiqueta Nodo,Nombre Nodo,PosiciÃ³n,Coordenada X,Coordenada Y,Atributos
0,URBANO,10184,740,1,2.0,338.0,10415.0,740_V1,Circular,52845.0,247A05_TM,247A05_Br. La Esperanza II,0.0,595348.0,521193.0,"SinÃ³ptico, Horario GOAL, Punto de control (TM..."
1,URBANO,10184,740,1,2.0,338.0,10415.0,740_V1,Circular,52806.0,222A05_TM,222A05_Hospital de EngativÃ¡ EmaÃºs,138.0,595258.0,521149.0,NaN


In [280]:
md = md[
    md["Id Nodo"].notna() &
    (md["Id Nodo"].astype(str).str.strip() != "") &
    (~md["Id Nodo"].astype(str).str.lower().isin(["nan"]))
]

md["Id Nodo"] = md["Id Nodo"].astype(int)
md["Id Ruta"] = md["Id Ruta"].astype(int)

md.head()

,ï»¿Tipo de Servicio,Id LÃ­nea,LÃ­nea,Configuraciones,ConfiguraciÃ³n Activa,Id SublÃ­nea,Id Ruta,Nombre Ruta,Sentido,Id Nodo,Etiqueta Nodo,Nombre Nodo,PosiciÃ³n,Coordenada X,Coordenada Y,Atributos
0,URBANO,10184,740,1,2.0,338.0,10415,740_V1,Circular,52845,247A05_TM,247A05_Br. La Esperanza II,0.0,595348.0,521193.0,"SinÃ³ptico, Horario GOAL, Punto de control (TM..."
1,URBANO,10184,740,1,2.0,338.0,10415,740_V1,Circular,52806,222A05_TM,222A05_Hospital de EngativÃ¡ EmaÃºs,138.0,595258.0,521149.0,NaN
2,URBANO,10184,740,1,2.0,338.0,10415,740_V1,Circular,52365,068A05_TM,068A05_Liceo SalomÃ³n Sabio,425.0,595014.0,520997.0,NaN
3,URBANO,10184,740,1,2.0,338.0,10415,740_V1,Circular,52442,110A05_TM,110A05_Br. Sabana del Dorado,686.0,594933.0,520819.0,NaN
4,URBANO,10184,740,1,2.0,338.0,10415,740_V1,Circular,52369,070A05_TM,070A05_Br. Sabana del Dorado,964.0,595091.0,520593.0,NaN


In [281]:
#renombrar nombres de columnas

md = md.rename(columns={
    "Id LÃ­nea": "Id Línea",
    "PosiciÃ³n": "Posición"
})

md.head(3)

,ï»¿Tipo de Servicio,Id Línea,LÃ­nea,Configuraciones,ConfiguraciÃ³n Activa,Id SublÃ­nea,Id Ruta,Nombre Ruta,Sentido,Id Nodo,Etiqueta Nodo,Nombre Nodo,Posición,Coordenada X,Coordenada Y,Atributos
0,URBANO,10184,740,1,2.0,338.0,10415,740_V1,Circular,52845,247A05_TM,247A05_Br. La Esperanza II,0.0,595348.0,521193.0,"SinÃ³ptico, Horario GOAL, Punto de control (TM..."
1,URBANO,10184,740,1,2.0,338.0,10415,740_V1,Circular,52806,222A05_TM,222A05_Hospital de EngativÃ¡ EmaÃºs,138.0,595258.0,521149.0,NaN
2,URBANO,10184,740,1,2.0,338.0,10415,740_V1,Circular,52365,068A05_TM,068A05_Liceo SalomÃ³n Sabio,425.0,595014.0,520997.0,NaN


In [282]:
md = md.sort_values(
    by=['Id Línea', 'Id Ruta', 'Posición']
).reset_index(drop=True)

md.head(2)

,ï»¿Tipo de Servicio,Id Línea,LÃ­nea,Configuraciones,ConfiguraciÃ³n Activa,Id SublÃ­nea,Id Ruta,Nombre Ruta,Sentido,Id Nodo,Etiqueta Nodo,Nombre Nodo,Posición,Coordenada X,Coordenada Y,Atributos
0,PADRON,10006,M84-C84,1,5.0,13.0,10013,M84 - C84_V1,Circular,71475,193B03,Av Suba - K114D,0.0,599402.0,525137.0,"SinÃ³ptico, Horario GOAL, Punto de control (TM..."
1,PADRON,10006,M84-C84,1,5.0,13.0,10013,M84 - C84_V1,Circular,71476,195B03,Av Suba - K110A.,453.0,599813.0,524944.0,"SinÃ³ptico, Horario GOAL, Punto de control (TM..."


In [283]:
md['orden'] = md.groupby(
    ['Id Línea', 'Id Ruta']
).cumcount() + 1

md.head(2)

,ï»¿Tipo de Servicio,Id Línea,LÃ­nea,Configuraciones,ConfiguraciÃ³n Activa,Id SublÃ­nea,Id Ruta,Nombre Ruta,Sentido,Id Nodo,Etiqueta Nodo,Nombre Nodo,Posición,Coordenada X,Coordenada Y,Atributos,orden
0,PADRON,10006,M84-C84,1,5.0,13.0,10013,M84 - C84_V1,Circular,71475,193B03,Av Suba - K114D,0.0,599402.0,525137.0,"SinÃ³ptico, Horario GOAL, Punto de control (TM...",1
1,PADRON,10006,M84-C84,1,5.0,13.0,10013,M84 - C84_V1,Circular,71476,195B03,Av Suba - K110A.,453.0,599813.0,524944.0,"SinÃ³ptico, Horario GOAL, Punto de control (TM...",2


In [284]:
md["Parada"] = md["Etiqueta Nodo"].str.split("_").str[0]

md.head()

,ï»¿Tipo de Servicio,Id Línea,LÃ­nea,Configuraciones,ConfiguraciÃ³n Activa,Id SublÃ­nea,Id Ruta,Nombre Ruta,Sentido,Id Nodo,Etiqueta Nodo,Nombre Nodo,Posición,Coordenada X,Coordenada Y,Atributos,orden,Parada
0,PADRON,10006,M84-C84,1,5.0,13.0,10013,M84 - C84_V1,Circular,71475,193B03,Av Suba - K114D,0.0,599402.0,525137.0,"SinÃ³ptico, Horario GOAL, Punto de control (TM...",1,193B03
1,PADRON,10006,M84-C84,1,5.0,13.0,10013,M84 - C84_V1,Circular,71476,195B03,Av Suba - K110A.,453.0,599813.0,524944.0,"SinÃ³ptico, Horario GOAL, Punto de control (TM...",2,195B03
2,PADRON,10006,M84-C84,1,5.0,13.0,10013,M84 - C84_V1,Circular,61413,C004,21 Ãngeles A - 2,3097.0,601949.0,523468.0,"SinÃ³ptico, Punto de control (TM-GOAL)",3,C004
3,PADRON,10006,M84-C84,1,5.0,13.0,10013,M84 - C84_V1,Circular,61441,C032,Puentelargo A - 2,8326.0,603467.0,518807.0,SinÃ³ptico,4,C032
4,PADRON,10006,M84-C84,1,5.0,13.0,10013,M84 - C84_V1,Circular,61523,D054,Polo C - 2 Ã³ 5,11858.0,603726.0,516325.0,"SinÃ³ptico, Punto de control (TM-GOAL)",5,D054


Actividad de bus

In [285]:
#Abrir Actividad de bus

archivo = f"Z:/01 base_datos/03 actividad_bus_FMS/{dia}_actividad_bus.csv"

# Cargar el archivo 

ab = pd.read_csv(archivo, encoding="latin")

#renombrar nombres de columnas

ab = ab.rename(columns={
    "Id LÃ­nea": "Id Línea"
})

ab.head(3)

C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_22108\1362021397.py:7: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  ab = pd.read_csv(archivo, encoding="latin")


,Fecha,Concesión,Concesionario de Operación,Id Línea,Línea,Id Ruta,Ruta,Tabla,Viaje Linea,Orden Viaje,...,Nombre de Conductor,Evento,Hora Teórica,Hora Referencia,Hora Llegada,Hora Salida,Tiempo entre paradas,Alarma Exceso de Tiempo en Parada,Tiempo Apertura de Puertas,Lista de acciones regulatorias
0,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10311,DA213,12730,DA213_V2,13,1,1,...,JOHN JAIRO MORENO HUERFANO,Inicio Viaje(3),7:00:00,7:00:00,NaN,NaN,NaN,NaN,NaN,NaN
1,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10311,DA213,12730,DA213_V2,13,1,1,...,JOHN JAIRO MORENO HUERFANO,NaN,7:00:50,7:00:50,NaN,NaN,NaN,NaN,NaN,NaN
2,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10311,DA213,12730,DA213_V2,13,1,1,...,JOHN JAIRO MORENO HUERFANO,NaN,7:02:12,7:02:12,NaN,NaN,NaN,NaN,NaN,NaN


In [286]:
data = ab.copy()

In [287]:
# Filtrar las filas donde 'HoraLlegada' no sea NaT (datos no vacíos)
data = data.dropna(subset=['Hora Llegada'])

data.head(3)

,Fecha,Concesión,Concesionario de Operación,Id Línea,Línea,Id Ruta,Ruta,Tabla,Viaje Linea,Orden Viaje,...,Nombre de Conductor,Evento,Hora Teórica,Hora Referencia,Hora Llegada,Hora Salida,Tiempo entre paradas,Alarma Exceso de Tiempo en Parada,Tiempo Apertura de Puertas,Lista de acciones regulatorias
98,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10311,DA213,12730,DA213_V2,13,1,1,...,JOHN JAIRO MORENO HUERFANO,NaN,9:31:39,9:31:39,9:12:29,9:12:29,NaN,NaN,0:00:00,NaN
99,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10311,DA213,12730,DA213_V2,13,1,1,...,JOHN JAIRO MORENO HUERFANO,NaN,9:32:11,9:32:11,9:12:42,9:13:02,0:00:13,NaN,0:00:00,NaN
100,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10311,DA213,12730,DA213_V2,13,1,1,...,JOHN JAIRO MORENO HUERFANO,NaN,9:33:44,9:33:44,9:14:24,9:14:43,0:01:22,NaN,0:00:00,NaN


In [288]:
data['HoraReferencia_m'] = data['Hora Referencia']
data['HoraLlegada_m'] = data['Hora Llegada']
data['Hora Teórica_m'] = data['Hora Teórica']

data.head(3)

,Fecha,Concesión,Concesionario de Operación,Id Línea,Línea,Id Ruta,Ruta,Tabla,Viaje Linea,Orden Viaje,...,Hora Referencia,Hora Llegada,Hora Salida,Tiempo entre paradas,Alarma Exceso de Tiempo en Parada,Tiempo Apertura de Puertas,Lista de acciones regulatorias,HoraReferencia_m,HoraLlegada_m,Hora Teórica_m
98,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10311,DA213,12730,DA213_V2,13,1,1,...,9:31:39,9:12:29,9:12:29,NaN,NaN,0:00:00,NaN,9:31:39,9:12:29,9:31:39
99,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10311,DA213,12730,DA213_V2,13,1,1,...,9:32:11,9:12:42,9:13:02,0:00:13,NaN,0:00:00,NaN,9:32:11,9:12:42,9:32:11
100,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10311,DA213,12730,DA213_V2,13,1,1,...,9:33:44,9:14:24,9:14:43,0:01:22,NaN,0:00:00,NaN,9:33:44,9:14:24,9:33:44


In [289]:
# Función para corregir horas con "24:00:00" o mayores
def fix_time(date_str):
    try:
        if '24:' in date_str:
            date_str = date_str.replace('24:', '00:')
            return pd.to_datetime(date_str, format='%H:%M:%S') + pd.Timedelta(days=1)
        elif '25:' in date_str:
            date_str = date_str.replace('25:', '01:')
            return pd.to_datetime(date_str, format='%H:%M:%S') + pd.Timedelta(days=1)
        elif '26:' in date_str:
            date_str = date_str.replace('26:', '02:')
            return pd.to_datetime(date_str, format='%H:%M:%S') + pd.Timedelta(days=1)
        elif '27:' in date_str:
            date_str = date_str.replace('27:', '03:')
            return pd.to_datetime(date_str, format='%H:%M:%S') + pd.Timedelta(days=1)
        elif '28:' in date_str:
            date_str = date_str.replace('28:', '04:')
            return pd.to_datetime(date_str, format='%H:%M:%S') + pd.Timedelta(days=1)
        elif '29:' in date_str:
            date_str = date_str.replace('29:', '05:')
            return pd.to_datetime(date_str, format='%H:%M:%S') + pd.Timedelta(days=1)
        else:
            return pd.to_datetime(date_str, format='%H:%M:%S')
    except (ValueError, TypeError) as e:
        print(f"Error al procesar '{date_str}': {e}")
        return pd.NaT  # Devolver NaT si hay un error

# Función para convertir tiempo a segundos
def time_to_seconds(time_obj):
    if pd.isnull(time_obj):
        return None
    return time_obj.hour * 3600 + time_obj.minute * 60 + time_obj.second

# Convertir las columnas de tiempo a cadenas si es necesario
data['HoraLlegada_m'] = data['HoraLlegada_m'].astype(str)
data['HoraReferencia_m'] = data['HoraReferencia_m'].astype(str)
data['Hora Teórica_m'] = data['Hora Teórica_m'].astype(str)

# Aplicar la función para corregir los tiempos en 'HoraLlegada' y 'HoraReferencia'
data['HoraLlegada_m'] = data['HoraLlegada_m'].apply(fix_time)
data['HoraReferencia_m'] = data['HoraReferencia_m'].apply(fix_time)
data['Hora Teórica_m'] = data['Hora Teórica_m'].apply(fix_time)

# Convertir las columnas 'HoraReferencia' y 'HoraLlegada' a segundos
data['HoraReferencia_segundos_m'] = data['HoraReferencia_m'].apply(time_to_seconds)
data['HoraLlegada_segundos_m'] = data['HoraLlegada_m'].apply(time_to_seconds)
data['Hora Teórica_segundos_m'] = data['Hora Teórica_m'].apply(time_to_seconds)

# Calcular la diferencia en segundos entre 'HoraLlegada' y 'HoraReferencia'
data['Diferencia_segundos_m'] = data['HoraLlegada_segundos_m'] - data['HoraReferencia_segundos_m']
data['Diferencia_segundos_ms'] = data['Hora Teórica_segundos_m'] - data['HoraReferencia_segundos_m']

# Mostrar el DataFrame actualizado
data.head(3)

,Fecha,Concesión,Concesionario de Operación,Id Línea,Línea,Id Ruta,Ruta,Tabla,Viaje Linea,Orden Viaje,...,Tiempo Apertura de Puertas,Lista de acciones regulatorias,HoraReferencia_m,HoraLlegada_m,Hora Teórica_m,HoraReferencia_segundos_m,HoraLlegada_segundos_m,Hora Teórica_segundos_m,Diferencia_segundos_m,Diferencia_segundos_ms
98,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10311,DA213,12730,DA213_V2,13,1,1,...,0:00:00,NaN,1900-01-01 09:31:39,1900-01-01 09:12:29,1900-01-01 09:31:39,34299,33149,34299,-1150,0
99,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10311,DA213,12730,DA213_V2,13,1,1,...,0:00:00,NaN,1900-01-01 09:32:11,1900-01-01 09:12:42,1900-01-01 09:32:11,34331,33162,34331,-1169,0
100,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10311,DA213,12730,DA213_V2,13,1,1,...,0:00:00,NaN,1900-01-01 09:33:44,1900-01-01 09:14:24,1900-01-01 09:33:44,34424,33264,34424,-1160,0


In [290]:
# Rellena los valores NaN con 0
data['Conductor'].fillna(0, inplace=True)

#Convertir la columna a tipo entero
data['Conductor'] = data['Conductor'].astype(int)

data.head(3)

C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_22108\2024684006.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['Conductor'].fillna(0, inplace=True)


,Fecha,Concesión,Concesionario de Operación,Id Línea,Línea,Id Ruta,Ruta,Tabla,Viaje Linea,Orden Viaje,...,Tiempo Apertura de Puertas,Lista de acciones regulatorias,HoraReferencia_m,HoraLlegada_m,Hora Teórica_m,HoraReferencia_segundos_m,HoraLlegada_segundos_m,Hora Teórica_segundos_m,Diferencia_segundos_m,Diferencia_segundos_ms
98,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10311,DA213,12730,DA213_V2,13,1,1,...,0:00:00,NaN,1900-01-01 09:31:39,1900-01-01 09:12:29,1900-01-01 09:31:39,34299,33149,34299,-1150,0
99,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10311,DA213,12730,DA213_V2,13,1,1,...,0:00:00,NaN,1900-01-01 09:32:11,1900-01-01 09:12:42,1900-01-01 09:32:11,34331,33162,34331,-1169,0
100,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10311,DA213,12730,DA213_V2,13,1,1,...,0:00:00,NaN,1900-01-01 09:33:44,1900-01-01 09:14:24,1900-01-01 09:33:44,34424,33264,34424,-1160,0


In [291]:
# 1. Asegurar mismos tipos (MUY importante)
for col in ['Id Línea', 'Id Ruta', 'Id Nodo']:
    data[col] = data[col].astype(str).str.strip()
    md[col] = md[col].astype(str).str.strip()

# 2. Eliminar duplicados en md (clave única)
md = md.drop_duplicates(subset=['Id Línea', 'Id Ruta', 'Id Nodo'])

# 3. Crear clave compuesta en md
md_dict = {
    (fila['Id Línea'], fila['Id Ruta'], fila['Id Nodo']): fila['Posición']
    for _, fila in md.iterrows()
}

# 4. Asignar Posición en data usando esa clave
data['Posición'] = data.apply(
    lambda x: md_dict.get((x['Id Línea'], x['Id Ruta'], x['Id Nodo'])),
    axis=1
)

data['Posición'] = pd.to_numeric(data['Posición'], errors='coerce').astype('Int64')

data = data.rename(columns={
    "Posición": "Posicion"
})

data.head(3)

,Fecha,Concesión,Concesionario de Operación,Id Línea,Línea,Id Ruta,Ruta,Tabla,Viaje Linea,Orden Viaje,...,Lista de acciones regulatorias,HoraReferencia_m,HoraLlegada_m,Hora Teórica_m,HoraReferencia_segundos_m,HoraLlegada_segundos_m,Hora Teórica_segundos_m,Diferencia_segundos_m,Diferencia_segundos_ms,Posicion
98,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10311,DA213,12730,DA213_V2,13,1,1,...,NaN,1900-01-01 09:31:39,1900-01-01 09:12:29,1900-01-01 09:31:39,34299,33149,34299,-1150,0,33733
99,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10311,DA213,12730,DA213_V2,13,1,1,...,NaN,1900-01-01 09:32:11,1900-01-01 09:12:42,1900-01-01 09:32:11,34331,33162,34331,-1169,0,33851
100,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10311,DA213,12730,DA213_V2,13,1,1,...,NaN,1900-01-01 09:33:44,1900-01-01 09:14:24,1900-01-01 09:33:44,34424,33264,34424,-1160,0,34195


In [292]:
# 1. Asegurar mismos tipos (MUY importante)
for col in ['Id Línea', 'Id Ruta', 'Id Nodo']:
    data[col] = data[col].astype(str).str.strip()
    md[col] = md[col].astype(str).str.strip()

# 2. Eliminar duplicados en md (clave única)
md = md.drop_duplicates(subset=['Id Línea', 'Id Ruta', 'Id Nodo'])

# 3. Crear clave compuesta en md
md_dict = {
    (fila['Id Línea'], fila['Id Ruta'], fila['Id Nodo']): fila['orden']
    for _, fila in md.iterrows()
}

# 4. Asignar Posición en data usando esa clave
data['orden'] = data.apply(
    lambda x: md_dict.get((x['Id Línea'], x['Id Ruta'], x['Id Nodo'])),
    axis=1
)

data['orden'] = pd.to_numeric(data['orden'], errors='coerce').astype('Int64')

data.head(3)

,Fecha,Concesión,Concesionario de Operación,Id Línea,Línea,Id Ruta,Ruta,Tabla,Viaje Linea,Orden Viaje,...,HoraReferencia_m,HoraLlegada_m,Hora Teórica_m,HoraReferencia_segundos_m,HoraLlegada_segundos_m,Hora Teórica_segundos_m,Diferencia_segundos_m,Diferencia_segundos_ms,Posicion,orden
98,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10311,DA213,12730,DA213_V2,13,1,1,...,1900-01-01 09:31:39,1900-01-01 09:12:29,1900-01-01 09:31:39,34299,33149,34299,-1150,0,33733,99
99,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10311,DA213,12730,DA213_V2,13,1,1,...,1900-01-01 09:32:11,1900-01-01 09:12:42,1900-01-01 09:32:11,34331,33162,34331,-1169,0,33851,100
100,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10311,DA213,12730,DA213_V2,13,1,1,...,1900-01-01 09:33:44,1900-01-01 09:14:24,1900-01-01 09:33:44,34424,33264,34424,-1160,0,34195,101


In [293]:
# 1. Asegurar mismos tipos (MUY importante)
for col in ['Id Línea', 'Id Ruta', 'Id Nodo']:
    data[col] = data[col].astype(str).str.strip()
    md[col] = md[col].astype(str).str.strip()

# 2. Eliminar duplicados en md (clave única)
md = md.drop_duplicates(subset=['Id Línea', 'Id Ruta', 'Id Nodo'])

# 3. Crear clave compuesta en md
md_dict = {
    (fila['Id Línea'], fila['Id Ruta'], fila['Id Nodo']): fila['Parada']
    for _, fila in md.iterrows()
}

# 4. Asignar Posición en data usando esa clave
data['Parada'] = data.apply(
    lambda x: md_dict.get((x['Id Línea'], x['Id Ruta'], x['Id Nodo'])),
    axis=1
)

data.head(3)

,Fecha,Concesión,Concesionario de Operación,Id Línea,Línea,Id Ruta,Ruta,Tabla,Viaje Linea,Orden Viaje,...,HoraLlegada_m,Hora Teórica_m,HoraReferencia_segundos_m,HoraLlegada_segundos_m,Hora Teórica_segundos_m,Diferencia_segundos_m,Diferencia_segundos_ms,Posicion,orden,Parada
98,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10311,DA213,12730,DA213_V2,13,1,1,...,1900-01-01 09:12:29,1900-01-01 09:31:39,34299,33149,34299,-1150,0,33733,99,043A05
99,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10311,DA213,12730,DA213_V2,13,1,1,...,1900-01-01 09:12:42,1900-01-01 09:32:11,34331,33162,34331,-1169,0,33851,100,237A05
100,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10311,DA213,12730,DA213_V2,13,1,1,...,1900-01-01 09:14:24,1900-01-01 09:33:44,34424,33264,34424,-1160,0,34195,101,241A05


In [294]:
#Ordenar por linea_x, ruta, descripcion, hora llegada para identificar el intervalo real

data = data.sort_values(by=['Id Línea','Id Ruta','Posicion','Hora Llegada'])

data.head(3)

,Fecha,Concesión,Concesionario de Operación,Id Línea,Línea,Id Ruta,Ruta,Tabla,Viaje Linea,Orden Viaje,...,HoraLlegada_m,Hora Teórica_m,HoraReferencia_segundos_m,HoraLlegada_segundos_m,Hora Teórica_segundos_m,Diferencia_segundos_m,Diferencia_segundos_ms,Posicion,orden,Parada
233574,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,16,1,1,...,1900-01-01 10:11:22,1900-01-01 09:15:45,30645,36682,33345,6037,2700,0,1,247A05
233914,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,17,1,1,...,1900-01-01 10:13:17,1900-01-01 09:21:30,30990,36797,33690,5807,2700,0,1,247A05
236804,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,22,1,1,...,1900-01-01 10:20:26,1900-01-01 10:08:15,33795,37226,36495,3431,2700,0,1,247A05


In [295]:
# Función para corregir las horas
def corregir_horas(hora):
    try:
        partes = hora.split(':')
        horas = int(partes[0])
        if horas >= 24:
            horas = horas % 24
        return f'{horas:02d}:{partes[1]}:{partes[2]}'
    except AttributeError:
        return hora  # Maneja casos donde el valor no es una cadena
    except ValueError:
        return hora  # Maneja casos donde el valor no se puede convertir a entero

# Convertir la columna 'Horas' a string
data['Hora Llegada'] = data['Hora Llegada'].astype(str)

# Aplicar la función a la columna 'Horas'
data['Hora Llegada'] = data['Hora Llegada'].apply(corregir_horas)

data.head(3)

,Fecha,Concesión,Concesionario de Operación,Id Línea,Línea,Id Ruta,Ruta,Tabla,Viaje Linea,Orden Viaje,...,HoraLlegada_m,Hora Teórica_m,HoraReferencia_segundos_m,HoraLlegada_segundos_m,Hora Teórica_segundos_m,Diferencia_segundos_m,Diferencia_segundos_ms,Posicion,orden,Parada
233574,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,16,1,1,...,1900-01-01 10:11:22,1900-01-01 09:15:45,30645,36682,33345,6037,2700,0,1,247A05
233914,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,17,1,1,...,1900-01-01 10:13:17,1900-01-01 09:21:30,30990,36797,33690,5807,2700,0,1,247A05
236804,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,22,1,1,...,1900-01-01 10:20:26,1900-01-01 10:08:15,33795,37226,36495,3431,2700,0,1,247A05


In [296]:
# Calcula la diferencia solo si las filas son iguales en las columnas 'linea_X', 'Ruta' y 'Posicion'
data['Intervalos'] = data.groupby(['Id Línea', 'Id Ruta', 'Posicion'])['HoraLlegada_segundos_m'].diff()

data['Intervalos'] = data['Intervalos'].round(3)

# Rellena los valores NaN (debido a la diferencia) con un espacio en blanco o cualquier otro valor
data['Intervalos'].fillna('', inplace=True)

data.head(3)

C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_22108\205850429.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['Intervalos'].fillna('', inplace=True)
C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_22108\205850429.py:7: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  data['Intervalos'].fillna('', inplace=True)


,Fecha,Concesión,Concesionario de Operación,Id Línea,Línea,Id Ruta,Ruta,Tabla,Viaje Linea,Orden Viaje,...,Hora Teórica_m,HoraReferencia_segundos_m,HoraLlegada_segundos_m,Hora Teórica_segundos_m,Diferencia_segundos_m,Diferencia_segundos_ms,Posicion,orden,Parada,Intervalos
233574,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,16,1,1,...,1900-01-01 09:15:45,30645,36682,33345,6037,2700,0,1,247A05,
233914,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,17,1,1,...,1900-01-01 09:21:30,30990,36797,33690,5807,2700,0,1,247A05,115.0
236804,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,22,1,1,...,1900-01-01 10:08:15,33795,37226,36495,3431,2700,0,1,247A05,429.0


In [297]:
#Diferencia ejecutada (intervalo programado)
# Calcular la diferencia entre filas en la columna 'FechaHora' 
data['Diferencia_prog'] = data['Hora Teórica_segundos_m'].diff()
data['Diferencia_prog'] = data['Diferencia_prog'].round(3)

data.head()

,Fecha,Concesión,Concesionario de Operación,Id Línea,Línea,Id Ruta,Ruta,Tabla,Viaje Linea,Orden Viaje,...,HoraReferencia_segundos_m,HoraLlegada_segundos_m,Hora Teórica_segundos_m,Diferencia_segundos_m,Diferencia_segundos_ms,Posicion,orden,Parada,Intervalos,Diferencia_prog
233574,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,16,1,1,...,30645,36682,33345,6037,2700,0,1,247A05,,NaN
233914,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,17,1,1,...,30990,36797,33690,5807,2700,0,1,247A05,115.0,345.0
236804,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,22,1,1,...,33795,37226,36495,3431,2700,0,1,247A05,429.0,2805.0
244795,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,41,1,1,...,36345,37408,36345,1063,0,0,1,247A05,182.0,-150.0
234934,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,19,1,1,...,31680,36067,34380,4387,2700,0,1,247A05,-1341.0,-1965.0


In [298]:
#Diferencia ejecutada (intervalo ejecutado)
# Calcular la diferencia entre filas en la columna 'FechaHora' 
data['Diferencia'] = data['HoraLlegada_segundos_m'].diff()
data['Diferencia'] = data['Diferencia'].round(3)

data.head()

,Fecha,Concesión,Concesionario de Operación,Id Línea,Línea,Id Ruta,Ruta,Tabla,Viaje Linea,Orden Viaje,...,HoraLlegada_segundos_m,Hora Teórica_segundos_m,Diferencia_segundos_m,Diferencia_segundos_ms,Posicion,orden,Parada,Intervalos,Diferencia_prog,Diferencia
233574,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,16,1,1,...,36682,33345,6037,2700,0,1,247A05,,NaN,NaN
233914,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,17,1,1,...,36797,33690,5807,2700,0,1,247A05,115.0,345.0,115.0
236804,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,22,1,1,...,37226,36495,3431,2700,0,1,247A05,429.0,2805.0,429.0
244795,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,41,1,1,...,37408,36345,1063,0,0,1,247A05,182.0,-150.0,182.0
234934,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,19,1,1,...,36067,34380,4387,2700,0,1,247A05,-1341.0,-1965.0,-1341.0


In [299]:
# Filtrar las filas que cumplen la condición y asignar la diferencia en la columna 'Diferencia'
data['Diferencia'] = np.where(
    (data['Id Ruta'].shift(-1) == data['Id Ruta']) & (data['Nombre Nodo'].shift(-1) == data['Nombre Nodo']),
    data['Diferencia'],
    pd.NaT
)

data.head(3)

,Fecha,Concesión,Concesionario de Operación,Id Línea,Línea,Id Ruta,Ruta,Tabla,Viaje Linea,Orden Viaje,...,HoraLlegada_segundos_m,Hora Teórica_segundos_m,Diferencia_segundos_m,Diferencia_segundos_ms,Posicion,orden,Parada,Intervalos,Diferencia_prog,Diferencia
233574,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,16,1,1,...,36682,33345,6037,2700,0,1,247A05,,NaN,NaN
233914,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,17,1,1,...,36797,33690,5807,2700,0,1,247A05,115.0,345.0,115.0
236804,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,22,1,1,...,37226,36495,3431,2700,0,1,247A05,429.0,2805.0,429.0


Convoy

In [300]:
# Aplica las condiciones para etiquetar menor a 2 minutos

data['Estado_convoy'] = np.where(
    (data['Id Línea'] == data['Id Línea'].shift(1)) &
    (data['Id Ruta'] == data['Id Ruta'].shift(1)) &
    (data['Id Nodo'] == data['Id Nodo'].shift(1)) &
    (data['Diferencia'] < 120),
    'Convoy',
    'Ok'
)

data.head(3)

,Fecha,Concesión,Concesionario de Operación,Id Línea,Línea,Id Ruta,Ruta,Tabla,Viaje Linea,Orden Viaje,...,Hora Teórica_segundos_m,Diferencia_segundos_m,Diferencia_segundos_ms,Posicion,orden,Parada,Intervalos,Diferencia_prog,Diferencia,Estado_convoy
233574,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,16,1,1,...,33345,6037,2700,0,1,247A05,,NaN,NaN,Ok
233914,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,17,1,1,...,33690,5807,2700,0,1,247A05,115.0,345.0,115.0,Convoy
236804,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,22,1,1,...,36495,3431,2700,0,1,247A05,429.0,2805.0,429.0,Ok


In [301]:
#Estado de intervalo programado en paradas
condiciones = [
    (data['Diferencia_prog'] < 300),
    (data['Diferencia_prog'] >= 300) & (data['Diferencia_prog'] < 600),
    (data['Diferencia_prog'] >= 600) & (data['Diferencia_prog'] < 900),
    (data['Diferencia_prog'] >= 900) & (data['Diferencia_prog'] < 1500),
    (data['Diferencia_prog'] >= 1500) & (data['Diferencia_prog'] < 2100),
    (data['Diferencia_prog'] >= 2100) & (data['Diferencia_prog'] < 3600),
    (data['Diferencia_prog'] > 3600)
]

valores = [
    "I<5",
    ">=5 I <10",
    ">=10 I <15",
    ">=15 I <25",
    ">=25 I <35",
    ">=35 I <50",
    "I>60"
]

data['Estado_Intervalo_prog'] = np.select(condiciones, valores, default="Otros")

data.head()

,Fecha,Concesión,Concesionario de Operación,Id Línea,Línea,Id Ruta,Ruta,Tabla,Viaje Linea,Orden Viaje,...,Diferencia_segundos_m,Diferencia_segundos_ms,Posicion,orden,Parada,Intervalos,Diferencia_prog,Diferencia,Estado_convoy,Estado_Intervalo_prog
233574,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,16,1,1,...,6037,2700,0,1,247A05,,NaN,NaN,Ok,Otros
233914,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,17,1,1,...,5807,2700,0,1,247A05,115.0,345.0,115.0,Convoy,>=5 I <10
236804,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,22,1,1,...,3431,2700,0,1,247A05,429.0,2805.0,429.0,Ok,>=35 I <50
244795,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,41,1,1,...,1063,0,0,1,247A05,182.0,-150.0,182.0,Ok,I<5
234934,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,19,1,1,...,4387,2700,0,1,247A05,-1341.0,-1965.0,-1341.0,Convoy,I<5


In [302]:
#Estado de intervalo ejecutado
condiciones = [
    (data['Diferencia'] < 300),
    (data['Diferencia'] >= 300) & (data['Diferencia'] < 600),
    (data['Diferencia'] >= 600) & (data['Diferencia'] < 900),
    (data['Diferencia'] >= 900) & (data['Diferencia'] < 1500),
    (data['Diferencia'] >= 1500) & (data['Diferencia'] < 2100),
    (data['Diferencia'] >= 2100) & (data['Diferencia'] < 3600),
    (data['Diferencia'] > 3600)
]

valores = [
    "I<5",
    ">=5 I <10",
    ">=10 I <15",
    ">=15 I <25",
    ">=25 I <35",
    ">=35 I <50",
    "I>60"
]

data['Estado_Intervalo'] = np.select(condiciones, valores, default="Otros")

data.head()

,Fecha,Concesión,Concesionario de Operación,Id Línea,Línea,Id Ruta,Ruta,Tabla,Viaje Linea,Orden Viaje,...,Diferencia_segundos_ms,Posicion,orden,Parada,Intervalos,Diferencia_prog,Diferencia,Estado_convoy,Estado_Intervalo_prog,Estado_Intervalo
233574,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,16,1,1,...,2700,0,1,247A05,,NaN,NaN,Ok,Otros,Otros
233914,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,17,1,1,...,2700,0,1,247A05,115.0,345.0,115.0,Convoy,>=5 I <10,I<5
236804,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,22,1,1,...,2700,0,1,247A05,429.0,2805.0,429.0,Ok,>=35 I <50,>=5 I <10
244795,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,41,1,1,...,0,0,1,247A05,182.0,-150.0,182.0,Ok,I<5,I<5
234934,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,19,1,1,...,2700,0,1,247A05,-1341.0,-1965.0,-1341.0,Convoy,I<5,I<5


In [303]:
mapa_categoria = {
    "I<5": "Bueno",
    ">=5 I <10": "Manejable",
    ">=10 I <15": "Regular",
    ">=15 I <25": "Revisar",
    ">=25 I <35": "Critico",
    ">=35 I <50": "Malo",
    "I>60": "Indeseable"
}

data['Categoria_prog'] = data['Estado_Intervalo_prog'].map(mapa_categoria)
data['Categoria'] = data['Estado_Intervalo'].map(mapa_categoria)

data.head(3)

,Fecha,Concesión,Concesionario de Operación,Id Línea,Línea,Id Ruta,Ruta,Tabla,Viaje Linea,Orden Viaje,...,orden,Parada,Intervalos,Diferencia_prog,Diferencia,Estado_convoy,Estado_Intervalo_prog,Estado_Intervalo,Categoria_prog,Categoria
233574,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,16,1,1,...,1,247A05,,NaN,NaN,Ok,Otros,Otros,NaN,NaN
233914,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,17,1,1,...,1,247A05,115.0,345.0,115.0,Convoy,>=5 I <10,I<5,Manejable,Bueno
236804,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,22,1,1,...,1,247A05,429.0,2805.0,429.0,Ok,>=35 I <50,>=5 I <10,Malo,Manejable


In [304]:
#Ordenar por linea_x, ruta, descripcion, hora llegada para identificar el intervalo real

data = data.sort_values(by=['Id Línea','Id Ruta','Posicion','Hora Llegada'])

data.head()

,Fecha,Concesión,Concesionario de Operación,Id Línea,Línea,Id Ruta,Ruta,Tabla,Viaje Linea,Orden Viaje,...,orden,Parada,Intervalos,Diferencia_prog,Diferencia,Estado_convoy,Estado_Intervalo_prog,Estado_Intervalo,Categoria_prog,Categoria
242414,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,34,3,3,...,1,247A05,-84975.0,1170.0,-84975.0,Convoy,>=15 I <25,I<5,Revisar,Bueno
241054,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,31,3,3,...,1,247A05,10.0,-540.0,10.0,Convoy,I<5,I<5,Bueno,Bueno
240374,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,30,4,4,...,1,247A05,1136.0,1545.0,1136.0,Ok,>=25 I <35,>=15 I <25,Critico,Revisar
224055,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,1,247A05,8621.0,7080.0,8621.0,Ok,I>60,I>60,Indeseable,Indeseable
224735,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,2,1,1,...,1,247A05,537.0,375.0,537.0,Ok,>=5 I <10,>=5 I <10,Manejable,Manejable


In [305]:
data['Hora Llegada'] = pd.to_datetime(
    data['Hora Llegada'],
    errors='coerce'
).dt.time

C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_22108\1418775362.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data['Hora Llegada'] = pd.to_datetime(


In [306]:
from datetime import time

data['Numero1'] = 1
data = data.reset_index(drop=True)

for i in range(1, len(data)):
    if (
        pd.notna(data.at[i, 'Id Ruta']) and
        pd.notna(data.at[i-1, 'Id Ruta']) and
        pd.notna(data.at[i, 'Posicion']) and
        pd.notna(data.at[i-1, 'Posicion']) and
        pd.notna(data.at[i, 'Hora Llegada']) and

        data.at[i, 'Id Ruta'] == data.at[i-1, 'Id Ruta'] and
        data.at[i, 'Posicion'] == data.at[i-1, 'Posicion'] and
        data.at[i, 'Hora Llegada'] >= time(3, 0, 0)
    ):
        data.at[i, 'Numero1'] = data.at[i-1, 'Numero1'] + 1
        
data.head()

,Fecha,Concesión,Concesionario de Operación,Id Línea,Línea,Id Ruta,Ruta,Tabla,Viaje Linea,Orden Viaje,...,Parada,Intervalos,Diferencia_prog,Diferencia,Estado_convoy,Estado_Intervalo_prog,Estado_Intervalo,Categoria_prog,Categoria,Numero1
0,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,34,3,3,...,247A05,-84975.0,1170.0,-84975.0,Convoy,>=15 I <25,I<5,Revisar,Bueno,1
1,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,31,3,3,...,247A05,10.0,-540.0,10.0,Convoy,I<5,I<5,Bueno,Bueno,1
2,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,30,4,4,...,247A05,1136.0,1545.0,1136.0,Ok,>=25 I <35,>=15 I <25,Critico,Revisar,1
3,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,247A05,8621.0,7080.0,8621.0,Ok,I>60,I>60,Indeseable,Indeseable,2
4,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,2,1,1,...,247A05,537.0,375.0,537.0,Ok,>=5 I <10,>=5 I <10,Manejable,Manejable,3


In [307]:
# # Inicializar una columna 'Numero' con 1

# data['Numero1'] = 1

# # Restablecer el índice del DataFrame

# data = data.reset_index(drop=True)

# # Iterar a través de las filas

# for i in range(1, len(data)):
#     if (data['Id Ruta'][i] == data['Id Ruta'][i-1]) and (data['Posicion'][i] == data['Posicion'][i-1]) and (data['Hora Llegada'][i] >= '03:00:00'):
#         data.at[i, 'Numero1'] = data.at[i-1, 'Numero1'] + 1
        
# data.head()

In [308]:
#Estado del bus en servicio

condiciones = [
    (data['HoraLlegada_segundos_m'] > data['HoraReferencia_segundos_m']),  
    (data['HoraLlegada_segundos_m'] == data['HoraReferencia_segundos_m']), 
    (data['HoraLlegada_segundos_m'] < data['HoraReferencia_segundos_m'])  
]

etiquetas = ['Atrasado', 'A tiempo', 'Adelantado']

data['Estado'] = np.select(condiciones, etiquetas, default='Otro')

data.head()

,Fecha,Concesión,Concesionario de Operación,Id Línea,Línea,Id Ruta,Ruta,Tabla,Viaje Linea,Orden Viaje,...,Intervalos,Diferencia_prog,Diferencia,Estado_convoy,Estado_Intervalo_prog,Estado_Intervalo,Categoria_prog,Categoria,Numero1,Estado
0,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,34,3,3,...,-84975.0,1170.0,-84975.0,Convoy,>=15 I <25,I<5,Revisar,Bueno,1,Adelantado
1,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,31,3,3,...,10.0,-540.0,10.0,Convoy,I<5,I<5,Bueno,Bueno,1,Adelantado
2,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,30,4,4,...,1136.0,1545.0,1136.0,Ok,>=25 I <35,>=15 I <25,Critico,Revisar,1,Adelantado
3,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,8621.0,7080.0,8621.0,Ok,I>60,I>60,Indeseable,Indeseable,2,Atrasado
4,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,2,1,1,...,537.0,375.0,537.0,Ok,>=5 I <10,>=5 I <10,Manejable,Manejable,3,Atrasado


In [309]:
data['franja'] = pd.to_datetime(
    data['Hora Llegada'],
    errors='coerce'
).dt.hour

In [310]:
# Franjas y cantidad de buses por franja

# Dividir la columna 'hora' en partes usando ':', y seleccionar la primera parte (horas)
data['franja'] = data['Hora Llegada'].apply(lambda x: x.hour if pd.notna(x) else None)

# Usamos groupby para agrupar por 'Linea_x', 'Ruta', 'Posicion' y 'Franja',
# luego usamos count() para contar las filas en cada grupo
data['Conteo'] = data.groupby(['Id Línea', 'Id Ruta', 'Posicion', 'franja'])['franja'].transform('count')

data.head()

,Fecha,Concesión,Concesionario de Operación,Id Línea,Línea,Id Ruta,Ruta,Tabla,Viaje Linea,Orden Viaje,...,Diferencia,Estado_convoy,Estado_Intervalo_prog,Estado_Intervalo,Categoria_prog,Categoria,Numero1,Estado,franja,Conteo
0,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,34,3,3,...,-84975.0,Convoy,>=15 I <25,I<5,Revisar,Bueno,1,Adelantado,0,3.0
1,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,31,3,3,...,10.0,Convoy,I<5,I<5,Bueno,Bueno,1,Adelantado,0,3.0
2,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,30,4,4,...,1136.0,Ok,>=25 I <35,>=15 I <25,Critico,Revisar,1,Adelantado,0,3.0
3,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,8621.0,Ok,I>60,I>60,Indeseable,Indeseable,2,Atrasado,3,9.0
4,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,2,1,1,...,537.0,Ok,>=5 I <10,>=5 I <10,Manejable,Manejable,3,Atrasado,3,9.0


In [311]:
paradas["cenefa"] = paradas["cenefa"].astype(str).str.strip().str.upper()
data["Parada"] = data["Parada"].astype(str).str.strip().str.upper()

In [312]:
#Cruzar con datos de nodo y Numero_parada

def calcular_turno(parada):
    
    filtro = (
        (paradas['cenefa'] == parada)
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not paradas.loc[filtro].empty:
        # Obtener el primer valor
        paradas1 = paradas.loc[filtro, 'latitud'].iloc[0]
        return paradas1 if not pd.isna(paradas1) else None  
    
    return None  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
data['latitud'] = data.apply(
    lambda row: calcular_turno(
        row['Parada']
    ),
    axis=1
)

data.head(3)

,Fecha,Concesión,Concesionario de Operación,Id Línea,Línea,Id Ruta,Ruta,Tabla,Viaje Linea,Orden Viaje,...,Estado_convoy,Estado_Intervalo_prog,Estado_Intervalo,Categoria_prog,Categoria,Numero1,Estado,franja,Conteo,latitud
0,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,34,3,3,...,Convoy,>=15 I <25,I<5,Revisar,Bueno,1,Adelantado,0,3.0,4.714588
1,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,31,3,3,...,Convoy,I<5,I<5,Bueno,Bueno,1,Adelantado,0,3.0,4.714588
2,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,30,4,4,...,Ok,>=25 I <35,>=15 I <25,Critico,Revisar,1,Adelantado,0,3.0,4.714588


In [313]:
#Cruzar con datos de nodo y Numero_parada

def calcular_turno(parada):
    
    filtro = (
        (paradas['cenefa'] == parada)
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not paradas.loc[filtro].empty:
        # Obtener el primer valor
        paradas1 = paradas.loc[filtro, 'longitud'].iloc[0]
        return paradas1 if not pd.isna(paradas1) else None  
    
    return None  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
data['longitud'] = data.apply(
    lambda row: calcular_turno(
        row['Parada']
    ),
    axis=1
)

data.head()

,Fecha,Concesión,Concesionario de Operación,Id Línea,Línea,Id Ruta,Ruta,Tabla,Viaje Linea,Orden Viaje,...,Estado_Intervalo_prog,Estado_Intervalo,Categoria_prog,Categoria,Numero1,Estado,franja,Conteo,latitud,longitud
0,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,34,3,3,...,>=15 I <25,I<5,Revisar,Bueno,1,Adelantado,0,3.0,4.714588,-74.140514
1,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,31,3,3,...,I<5,I<5,Bueno,Bueno,1,Adelantado,0,3.0,4.714588,-74.140514
2,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,30,4,4,...,>=25 I <35,>=15 I <25,Critico,Revisar,1,Adelantado,0,3.0,4.714588,-74.140514
3,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,I>60,I>60,Indeseable,Indeseable,2,Atrasado,3,9.0,4.714588,-74.140514
4,15/04/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,2,1,1,...,>=5 I <10,>=5 I <10,Manejable,Manejable,3,Atrasado,3,9.0,4.714588,-74.140514


In [314]:
data.to_csv(f'C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/Estado_paradas/{dia}_seguimiento_tiempos_paradas.csv', index=False, sep=';')